In [1]:
import json

def process_jsonl(input_path, output_path, target_keywords):
    with open(input_path, "r") as fin, open(output_path, "w") as fout:
        for line in fin:
            example = json.loads(line)
            label_str = example.get("label", "").lower()
            label = int(label_str in target_keywords)

            processed = {
                "input_ids": example["tokens"],
                "attention_mask": [1] * len(example["tokens"]),
                "labels": label
            }
            fout.write(json.dumps(processed) + "\n")


TARGET_KEYWORDS = {"stop"}

process_jsonl("data/tokens_train.jsonl", "data/binary_train.jsonl", TARGET_KEYWORDS)
process_jsonl("data/tokens_val.jsonl", "data/binary_val.jsonl", TARGET_KEYWORDS)
process_jsonl("data/tokens_test.jsonl", "data/binary_test.jsonl", TARGET_KEYWORDS)

In [2]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "data/binary_train.jsonl",
    "val": "data/binary_val.jsonl",
    "test": "data/binary_test.jsonl"
})

print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

{'input_ids': [17, 17, 296, 296, 296, 296, 20, 20, 320, 320, 219, 219, 357, 357, 357, 443, 443, 120, 271, 271, 150, 150, 150, 39, 390, 390, 390, 390, 390, 390, 18, 18, 18, 18, 112, 112, 56, 237, 442, 442, 442, 442, 442, 442, 442, 442, 442, 442, 17], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 0}


In [3]:
from transformers import GPT2Config, GPT2Tokenizer, GPT2ForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
import torch
import torch.nn as nn

# 1) 加载 tokenizer & 模型
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2,               # 二分类
    # pad_token_id=tokenizer.eos_token_id
    pad_token_id=0
)

HUBERT_VOCAB_SIZE = 500
HIDDEN_SIZE = model.transformer.wte.embedding_dim  # GPT2 中为 768
model.transformer.wte = nn.Embedding(HUBERT_VOCAB_SIZE, HIDDEN_SIZE)

# 2) 冻结原模型参数，只微调 LoRA 参数
for param in model.parameters():
    param.requires_grad = False

# 3) 配置 LoRA
lora_cfg = LoraConfig(
    r=8,                        # Low-rank 阶数
    lora_alpha=32,
    target_modules=["c_attn"],  # GPT-2 中作用于注意力的投射矩阵
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"         # 序列分类
)
model = get_peft_model(model, lora_cfg)

# 单独解冻 embedding 层
for param in model.base_model.model.transformer.wte.parameters():
    param.requires_grad = True


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Tools\Anaconda3\envs\speech\lib\site-packages\peft\tuners\lora\layer.py:1150: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [4]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [5]:
training_args = TrainingArguments(
    output_dir="./lora-gpt2-keyword",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    learning_rate=1e-4,
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    save_steps=200,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    tokenizer=tokenizer,
    data_collator=lambda data: {
        "input_ids":       torch.nn.utils.rnn.pad_sequence(
                                [torch.tensor(x["input_ids"]) for x in data],
                                batch_first=True,
                                # padding_value=tokenizer.pad_token_id),
                                padding_value=0),
        "attention_mask":  torch.nn.utils.rnn.pad_sequence(
                                [torch.tensor(x["attention_mask"]) for x in data],
                                batch_first=True,
                                padding_value=0),
        "labels":          torch.tensor([x["labels"] for x in data])
    },
    compute_metrics=compute_metrics
)

for name, param in model.named_parameters():
    if not param.requires_grad:
        print(f"[FROZEN] {name}")
    else:
        print(f"[TRAINABLE] {name}")

trainer.train()


C:\Tools\Anaconda3\envs\speech\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\qwert\AppData\Local\Temp\ipykernel_26028\3849103610.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


[TRAINABLE] base_model.model.transformer.wte.weight
[FROZEN] base_model.model.transformer.wpe.weight
[FROZEN] base_model.model.transformer.h.0.ln_1.weight
[FROZEN] base_model.model.transformer.h.0.ln_1.bias
[FROZEN] base_model.model.transformer.h.0.attn.c_attn.base_layer.weight
[FROZEN] base_model.model.transformer.h.0.attn.c_attn.base_layer.bias
[TRAINABLE] base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight
[TRAINABLE] base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight
[FROZEN] base_model.model.transformer.h.0.attn.c_proj.weight
[FROZEN] base_model.model.transformer.h.0.attn.c_proj.bias
[FROZEN] base_model.model.transformer.h.0.ln_2.weight
[FROZEN] base_model.model.transformer.h.0.ln_2.bias
[FROZEN] base_model.model.transformer.h.0.mlp.c_fc.weight
[FROZEN] base_model.model.transformer.h.0.mlp.c_fc.bias
[FROZEN] base_model.model.transformer.h.0.mlp.c_proj.weight
[FROZEN] base_model.model.transformer.h.0.mlp.c_proj.bias
[FROZEN] base_model.model.transform

C:\Tools\Anaconda3\envs\speech\lib\site-packages\transformers\models\gpt2\modeling_gpt2.py:545: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
200,0.332300,0.307448,0.899507,0.000000,0.000000,0.000000
400,0.201100,0.077885,0.980005,0.961310,0.834625,0.893499
600,0.055400,0.047837,0.988834,0.941026,0.948320,0.944659
800,0.040200,0.043234,0.988574,0.934177,0.953488,0.943734


C:\Tools\Anaconda3\envs\speech\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=964, training_loss=0.1590385370234731, metrics={'train_runtime': 351.5425, 'train_samples_per_second': 175.478, 'train_steps_per_second': 2.742, 'total_flos': 1548003070107648.0, 'train_loss': 0.1590385370234731, 'epoch': 2.0})

In [6]:
## tensorboard --logdir=D:\Backup\Workspace\000_Speech\project\lora-gpt2-keyword\runs

In [7]:
results = trainer.evaluate(dataset["test"])

In [8]:
print("Evaluation results on test set:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Evaluation results on test set:
eval_loss: 0.0357
eval_accuracy: 0.9907
eval_precision: 0.9512
eval_recall: 0.9561
eval_f1: 0.9536
eval_runtime: 8.2327
eval_samples_per_second: 467.7670
eval_steps_per_second: 7.4090
epoch: 2.0000


In [9]:
print(model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): GPT2ForSequenceClassification(
      (transformer): GPT2Model(
        (wte): Embedding(500, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2SdpaAttention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B

In [10]:
for name, param in model.named_parameters():
    if not param.requires_grad:
        print(f"[FROZEN] {name}")
    else:
        print(f"[TRAINABLE] {name}")

[TRAINABLE] base_model.model.transformer.wte.weight
[FROZEN] base_model.model.transformer.wpe.weight
[FROZEN] base_model.model.transformer.h.0.ln_1.weight
[FROZEN] base_model.model.transformer.h.0.ln_1.bias
[FROZEN] base_model.model.transformer.h.0.attn.c_attn.base_layer.weight
[FROZEN] base_model.model.transformer.h.0.attn.c_attn.base_layer.bias
[TRAINABLE] base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight
[TRAINABLE] base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight
[FROZEN] base_model.model.transformer.h.0.attn.c_proj.weight
[FROZEN] base_model.model.transformer.h.0.attn.c_proj.bias
[FROZEN] base_model.model.transformer.h.0.ln_2.weight
[FROZEN] base_model.model.transformer.h.0.ln_2.bias
[FROZEN] base_model.model.transformer.h.0.mlp.c_fc.weight
[FROZEN] base_model.model.transformer.h.0.mlp.c_fc.bias
[FROZEN] base_model.model.transformer.h.0.mlp.c_proj.weight
[FROZEN] base_model.model.transformer.h.0.mlp.c_proj.bias
[FROZEN] base_model.model.transform

In [11]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params} / {total_params} ({trainable_params/total_params:.2%})")

Trainable parameters: 680448 / 86524416 (0.79%)


In [12]:
from transformers import GPT2Config

# 1. 保存基础模型配置（包含修改后的vocab_size）
config = model.config
config.vocab_size = HUBERT_VOCAB_SIZE  # 确保配置反映实际修改
config.save_pretrained("custom_gpt2_model")

# # 2. 保存LoRA适配器配置
model.base_model.save_pretrained("custom_gpt2_model")

# 3. 保存模型权重（包括修改后的Embedding层和LoRA权重）
model.save_pretrained("custom_gpt2_model", safe_serialization=True)


# model.base_model.save_pretrained("./lora-gpt2-hubert-final")
# model.save_pretrained("./lora-gpt2-hubert-final/adapter")

C:\Tools\Anaconda3\envs\speech\lib\site-packages\peft\utils\save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
